# Inititial Pose Estimation of an Apple

## Imports
For the detection of the object a YOLO-model from Ultralytics is used. \
The 3D calculations are made with Open3D and OpenCV. \
For some little plotting tasks, matplotlib is used. \
The scanning data are saved as numpy arrays.

In [ ]:
# imports
import copy
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
import open3d as o3d
from open3d.web_visualizer import draw as web_draw
import sys
from ultralytics import YOLO

sys.path.append(os.path.abspath('../src'))
from camera import Camera
from evaluation import Evaluation
from paths import PathManager
from segmentation import predict_segment

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[Open3D INFO] Resetting default logger to print to terminal.


In [ ]:
# Settings for this notebook
apple_index = 0
web_visualization = False

In [ ]:
# draw function: depending on web_visualization
def draw(point_clouds):
    global web_visualization
    if web_visualization:
        web_draw(point_clouds)
    else:
        o3d.visualization.draw_geometries(point_clouds)

## Paths
The dataset is too big, to load direct into the GitHub. Because of that, I'll use an environment file on each device seperately.

In [ ]:
searched_item = 'apple'
pm = PathManager(searched_item)

## Images

### Color Image

The original camera images are taken as RGB pictures. \
To print them with pyplot is necessary to convert it to the BGR format.

In [ ]:
rgb_image = np.load(pm.get_rgb_image_path(apple_index))
bgr_image = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)
plt.imshow(bgr_image)
plt.title("Original Image")
plt.show()

### Depth image

The depth image is also saved as numpy array.

In [ ]:
depth_image = np.load(pm.get_depth_image_path(apple_index))
plt.imshow(depth_image, cmap="viridis")  # alternatives: 'gray' or 'plasma'
plt.title("Depth Image")
plt.colorbar()  # show colorbar
plt.show()

## YOLO 11

### Result Class

```python
for result in results:
    xy = result.masks.xy                # mask in polygon format
    xyn = result.masks.xyn              # normalized
    masks = result.masks.data           # mask in matrix format (num_objects x H x W)
    for box in result.boxes.data:       # iterate through bounding boxes
        class_id = int(box[-1])         # the last column is the class-id
        label = model.names[class_id]   # label from class-id
```

### Bounding Boxes and Segmentation

It is necessary to get a smaller working radius to get ICP work properly. \
If to many points are around, ICP will not work.

In [ ]:
pts, seg_image = predict_segment(rgb_image, searched_item)

In [ ]:
plt.imshow(cv2.cvtColor(seg_image, cv2.COLOR_BGR2RGB))  # Convert BGR to RGB for correct color representation
plt.title("Segmantation")
plt.show()

## Prepare Depth image

When the ICP algorithm should do a proper work, the area it is used, should be small. \
Because of that I use the segment of the object to get a smaller area.

The segmentation gets the smaller and more accurate area, the bounding box is ignored at this point.

The mask for the bounding box would be created the following way:
```python
mask = np.zeros_like(depth_image)       # Create a mask initialized with zeros (same shape as depth image)
mask[y1:y2, x1:x2] = 1                  # Set inside bounding box to 1 (keep those values)
masked_depth_image = depth_image * mask
```

In [ ]:
# Create a mask initialized with zeros (same shape as depth image)
mask = np.zeros_like(depth_image)
cv2.fillPoly(mask, [pts], color=1)  # Set inside polygon to 1
masked_depth_image = depth_image * mask

In [ ]:
# Show depth image
plt.imshow(masked_depth_image, cmap="viridis")
plt.title("masked depth image")
plt.colorbar()
plt.show()

## Camera Intrinsics and Images

```
fx = 607.2747802734375
fy = 607.3233032226562
cx = 318.98992919921875
cy = 248.74452209472656
```

The taken pictures are all the same size:
640x480 pixels

In [ ]:
cam = Camera.from_yaml("../data/camera_intrinsics.yaml", "D435i")

## Depth Image to Point Cloud

In [ ]:
def get_scene_point_cloud(depth_img, d_width, d_height, cam_params):
    (fx, fy, cx, cy) = cam_params
    depth_img = o3d.geometry.Image(depth_img)
    cam_intrinsics = o3d.camera.PinholeCameraIntrinsic(
        width=d_width,
        height=d_height,
        fx=fx,
        fy=fy,
        cx=cx,
        cy=cy)
    pcd = o3d.geometry.PointCloud.create_from_depth_image(
        depth=depth_img,
        intrinsic=cam_intrinsics,
        extrinsic=np.eye(4)
    )
    return pcd

In [ ]:
pcd = get_scene_point_cloud(depth_image, cam.width, cam.height, cam.get_camera_intrinsics())

In [ ]:
draw([pcd])

In [ ]:
cropped_pcd = get_scene_point_cloud(masked_depth_image, cam.width, cam.height, cam.get_camera_intrinsics())

In [ ]:
draw([cropped_pcd])

## 3D Position Estimation

### Intrinsic Parameters
The intrinsic parameters of a camera describe its optical properties and are essential for mapping 2D pixel coordinates to 3D world coordinates. These parameters typically include:
- Focal Length ((f_x, f_y)): Scaling factors along the x- and y-axes, measured in pixels.
- Principal Point ((c_x, c_y)): The image center in pixel coordinates.
- Distortion Parameters (optional): Corrections for lens distortions, such as radial or tangential distortions.

The intrinsic matrix (K) is defined as:
    $K = \begin{bmatrix}
    f_x & 0 & c_x \\
    0 & f_y & c_y \\
    0 & 0 & 1
    \end{bmatrix}$

The RGB-D camera provides a depth map, which assigns a depth value (z) to each pixel, representing the distance from the camera to the object along the optical axis.

### Centroid of Segmentation

Calculate the center of gravity (Centroid) of the segmentation mask in pixel coordinates. This is the weighted center point of the pixels that belong to the object:
$$x_c = \frac{\sum_{i} x_i \cdot M(i)}{\sum_{i} M(i)}, \quad y_c = \frac{\sum_{i} y_i \cdot M(i)}{\sum_{i} M(i)}$$
where $M(i)$ is the mask (1 for object pixels, 0 otherwise) and $(x_i, y_i)$ are the coordinates of the pixels.

### Determine depth value
Use the depth map to collect the depth values of all pixels within the segmentation mask. To minimize noise, you can use the median or the mean value of the depth values:

$$z = \text{Median}(\{z_i \mid M(i) = 1\}) \quad \text{or} \quad z = \frac{\sum_{i} z_i \cdot M(i)}{\sum_{i} M(i)}$$

### Conversion to 3D coordinates
Use the intrinsic matrix $K$ of the RGB-D camera to convert the center of gravity $(x_c, y_c)$ and the depth value $z$ into 3D camera coordinates $(X, Y, Z)$:
$$\begin{bmatrix}
X \\
Y \\
Z
\end{bmatrix}
=
\begin{bmatrix}
(x_c - c_x) \cdot \frac{z}{f_x} \\
(y_c - c_y) \cdot \frac{z}{f_y} \\
z
\end{bmatrix}$$


In [ ]:
position = cropped_pcd.get_center()
print(position)

In [ ]:
# euklidean distance
distance = np.linalg.norm(position)
print(distance)

## 3D-Model from DB

### Mesh model

In [ ]:
model = o3d.io.read_triangle_mesh(pm.get_model_path())

In [ ]:
model.compute_vertex_normals()
draw([model])

### Mesh to Pointcloud

To run ICP over the model, the mesh has to be converted into a point cloud.

For this there two possible functions from Open3D:
1. sample_points_uniformly
2. sample_points_poisson_disk

While the sample_points_uniformly is processed much faster, the sample_points_poisson_disk creates a more consistent point cloud.

In [ ]:
model_pcd = model.sample_points_poisson_disk(number_of_points=10000)
draw([model_pcd])

## Preparation of the point cloud
Preparation of the point cloud means to reduce potential error sources, such as outliers. 

In [ ]:
# create copy of cropped_pcd
filtered_pcd = copy.deepcopy(cropped_pcd)

In [ ]:
# remove outliers
filtered_pcd, ind = filtered_pcd.remove_statistical_outlier(nb_neighbors=50, std_ratio=0.7)

In [ ]:
# estimate normals
filtered_pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))

In [ ]:
draw([filtered_pcd])

In [ ]:
# show cropped pcd in original pcd
pcd_black = copy.deepcopy(pcd).paint_uniform_color([0,0,0])
draw([pcd_black, filtered_pcd])

## Initial Alignment

In [ ]:
print(f"Model center:       {model_pcd.get_center()}")
print(f"Point cloud center: {filtered_pcd.get_center()}")

### Calculate transformation matrix from center of model to center of scan

In [ ]:
T = np.eye(4)
T[:3, 3] = filtered_pcd.get_center() - model_pcd.get_center()
print(f"Translation: {T[:3, 3]}")   

In [ ]:
# transform with calculatet T
transformed_model = copy.deepcopy(model_pcd).transform(T)

### Preprocessing
In the preprocessing, the point clouds are downsampled with voxel, the normals are estimated and FPFH Features are computed. 

**FPFH:** Fast Point Feature Histograms - Descriptor for local geometrie around one point in a point cloud. 

In [ ]:
def preprocess_point_cloud(pcd, voxel_size):
    pcd_down = pcd.voxel_down_sample(voxel_size)
    pcd_down.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*2, max_nn=30)
    )
    fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down,
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*5, max_nn=100)
    )
    return pcd_down, fpfh

In [ ]:
voxel_size = 0.005 # in meters -> 5mm
scene_down, scene_fpfh = preprocess_point_cloud(filtered_pcd, voxel_size)
model_down, model_fpfh = preprocess_point_cloud(transformed_model, voxel_size)

print(scene_down, scene_fpfh)
print(model_down, model_fpfh)

### RANSAC

**RANSAC** (Random Sample Consensus) is a robust optimization algorithm used to estimate models from data containing many outliers. 
In 3D point cloud processing, RANSAC is commonly used to compute an initial rough alignment (transformation) between two point clouds.

#### Objective
Find the best transformation (rotation + translation) that explains as many valid point correspondences as possible between a source and a target point cloud – despite noise or outliers.

#### How It Works
Randomly sample $n$ point pairs (e.g., 3–4 correspondences). \
Compute transformation that best aligns these pairs. \
Apply transformation to the entire point cloud. \
Count inliers: points that align well within a threshold after transformation. \
Repeat many times → Keep the model with the most inliers. 

In [ ]:
result_ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    model_down,             # source
    scene_down,             # target
    model_fpfh,             # source feature
    scene_fpfh,             # target feature
    mutual_filter=False,    # mutual_filter
    max_correspondence_distance=voxel_size*1.5,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
    ransac_n=4,
    checkers=[
        o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
        o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(voxel_size*1.5),
    ],
    criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(4000000, 500)
)
print(result_ransac)

In [ ]:
draw([scene_down, model_down])

## ICP Algorithm

### Parameters

In [ ]:
source = copy.deepcopy(model_down)
target = copy.deepcopy(scene_down)
threshold = voxel_size

### o3d Pipeline

In [ ]:
reg_p2p = o3d.pipelines.registration.registration_icp(
    source=source, 
    target=target, 
    max_correspondence_distance=threshold,
    init=result_ransac.transformation,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(),
    criteria=o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=100),
)

print("Transformation matrix:")
print(reg_p2p.transformation)

In [ ]:
transformed_source = copy.deepcopy(source).transform(reg_p2p.transformation)
draw([transformed_source, target])

### Evaluation
$$\text{fitness} = \frac{\text{Anzahl der Inlier-Korrespondenzen}}{\text{Gesamtzahl der Punkte in der Quell-Punktewolke}}$$
$$\text{inlier\_rmse} = \sqrt{\frac{1}{N} \sum_{i=1}^N \|\text{p}_{source,i} - \text{p}_{target,i}\|^2}$$

$$\text{translation\_error} = \left\| \mathbf{t}_{\text{estimation}} - \mathbf{t}_{\text{ground\_truth}} \right\|_2 = \sqrt{(x_{\text{est}} - x_{\text{gt}})^2 + (y_{\text{est}} - y_{\text{gt}})^2 + (z_{\text{est}} - z_{\text{gt}})^2}$$
$$\theta = \cos^{-1} \left( \frac{\mathrm{trace} \left( R_{\text{est}}^\top R_{\text{gt}} \right) - 1}{2} \right) \Longrightarrow \text{rotation\_error} = \theta \cdot \frac{180^\circ}{\pi}$$

### Ground Truth

In [ ]:
T_gt = np.genfromtxt(pm.get_orientation_path(apple_index))
print("Ground Truth:\n", T_gt)

In [ ]:
eval = Evaluation(source, target, threshold, reg_p2p.transformation, T_gt)
print(eval)